# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("tic-tac-toe.csv")

target_col = df.columns[-1]

X_raw = df.drop(columns=[target_col])
y_raw = df[target_col]

ohe = OneHotEncoder(sparse_output=False)
X_ohe = ohe.fit_transform(X_raw)

y = (y_raw == "positive").astype(int).to_numpy()

scaler = StandardScaler()
X = scaler.fit_transform(X_ohe)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (958, 27)
y shape: (958,)


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(np.unique(y_train, return_counts=True))
print(np.unique(y_test, return_counts=True))

(766, 27) (192, 27)
(array([0]), array([766]))
(array([0]), array([192]))


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

n_features = X_train.shape[1]

model = Sequential([
    Dense(64, activation="relu", input_shape=(n_features,)),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(2, activation="softmax")
])

model.summary()

/opt/anaconda3/envs/ttt-tf/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-02 23:01:18.610392: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-03-02 23:01:18.610496: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-03-02 23:01:18.610520: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-03-02 23:01:18.610698: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-02 23:01:18.610724: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,434 (17.32 KB)

 Trainable params: 4,434 (17.32 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200


2026-03-02 23:02:34.382187: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 127ms/step - accuracy: 0.5408 - loss: 0.8370 - val_accuracy: 0.6494 - val_loss: 0.6785
Epoch 2/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.6879 - loss: 0.5851 - val_accuracy: 0.7662 - val_loss: 0.5177
Epoch 3/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7794 - loss: 0.4801 - val_accuracy: 0.9026 - val_loss: 0.3912
Epoch 4/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8513 - loss: 0.3941 - val_accuracy: 0.9481 - val_loss: 0.2941
Epoch 5/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8824 - loss: 0.3313 - val_accuracy: 1.0000 - val_loss: 0.2297
Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9216 - loss: 0.2771 - val_accuracy: 1.0000 - val_loss: 0.1754
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9150 - loss: 0.2502 - val_accuracy: 1.0000 - val_loss: 0.1390
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8562 - loss: 0.3811 - val_accuracy: 1.0000 - val_

In [13]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

Test loss: 1.2417632477834672e-09
Test accuracy: 1.0


In [15]:
model.save("tic-tac-toe.model.keras")
print("Saved to tic-tac-toe.model.keras")

Saved to tic-tac-toe.model.keras


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [17]:
from tensorflow.keras.models import load_model

model = load_model("tic-tac-toe.model.keras")
print("Model loaded successfully")

Model loaded successfully


In [18]:
import numpy as np

random_indices = np.random.choice(len(X_test), size=5, replace=False)

X_sample = X_test[random_indices]
y_sample = y_test[random_indices]

print("Selected indices:", random_indices)

Selected indices: [123 174  68   6  58]


In [19]:
y_proba = model.predict(X_sample)
y_pred = np.argmax(y_proba, axis=1)

print("Predicted classes:", y_pred)
print("Actual classes:   ", y_sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 734ms/step
Predicted classes: [0 0 0 0 0]
Actual classes:    [0 0 0 0 0]


In [20]:
correct = y_pred == y_sample
print("Correct predictions:", correct)

Correct predictions: [ True  True  True  True  True]


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

The model successfully achieved the required performance thresholds. The final test loss was significantly below 0.1, and the test accuracy reached 1.0, which is well above the required 0.95. Therefore, the objective of obtaining low loss and high accuracy was fully accomplished.

**Which approach(es) did you find helpful to improve your model performance?**

In [10]:
# your answer here